In [0]:
dbutils.widgets.text("process", "")

In [0]:
%run "./common functions"

In [0]:
process = dbutils.widgets.get("process")

In [0]:
spark.sql(f'truncate table staging_catalogue.jobs.{process}')
all_jobs = spark.table(f"raw_catalogue.jobs.{process}")
jobs = all_jobs.select("url","description","skills").collect()

In [0]:
display(jobs)

In [0]:
scored_jobs = score_job(jobs, "url", "description", "skills")

In [0]:
display(scored_jobs)

In [0]:
scored_df = spark.createDataFrame(scored_jobs)
df = scored_df.join(all_jobs, on=['url'])
filtered_df = df.filter(df.score > 70)
filtered_df.write.mode('overwrite').option("mergeSchema", "true").saveAsTable(f"staging_catalogue.jobs.{process}")

In [0]:
display(filtered_df)